<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_eda_and_preproc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: Análisis Exploratorio (EDA) y Carga de Datos
**Proyecto de Maestría:** Detección de Deslizamientos con IA
**Estudiante:** Lisa | Universidad EAFIT

Este notebook automatiza la preparación del entorno en Colab y la validación de los datos multiespectrales.

## 1. Configuración del Entorno
Instalamos librerías necesarias y preparamos el sistema para leer archivos H5.

In [ ]:
import os
import h5py
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files

print("✓ Librerías cargadas.")

## 2. Carga y Descompresión del Dataset
Sube el archivo `landslide4sense.zip` generado desde tu PC.

In [ ]:
# 2.1 Carga manual si no existe
path_zip = '/content/landslide4sense.zip'

if not os.path.exists(path_zip):
    print("Selecciona el archivo .zip en tu carpeta local:")
    uploaded = files.upload()
else:
    print("✓ El archivo zip ya está presente.")

# 2.2 Descompresión en carpeta temporal
with zipfile.ZipFile(path_zip, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset_temp')

# 2.3 Localización dinámica de la carpeta TrainData
# Buscamos el patrón que vimos en tu captura de pantalla
rutas_encontradas = list(Path('/content/dataset_temp').glob('**/TrainData/img'))

if rutas_encontradas:
    # DATA_ROOT será la carpeta que contiene TrainData, TestData y ValidData
    DATA_ROOT = rutas_encontradas[0].parent.parent
    print(f"\n✅ ¡Ruta detectada con éxito!")
    print(f"DATA_ROOT: {DATA_ROOT}")
    
    img_list = sorted(list((DATA_ROOT / "TrainData/img").glob("*.h5")))
    mask_list = sorted(list((DATA_ROOT / "TrainData/mask").glob("*.h5")))
    print(f"Total imágenes encontradas: {len(img_list)}")
else:
    print("❌ No se encontró 'TrainData/img'. Revisa la estructura del ZIP.")

## 3. Visualización y Análisis Multiespectral
Cargamos una muestra y visualizamos la combinación RGB de Sentinel-2.

In [ ]:
def read_h5(path):
    with h5py.File(path, 'r') as f:
        return np.array(f[list(f.keys())[0]])

# Tomar una muestra aleatoria
idx = np.random.randint(0, len(img_list))
img = read_h5(img_list[idx])
mask = read_h5(mask_list[idx])

# Extraer canales RGB (B4, B3, B2)
# El dataset suele tener el orden: [B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11, B12]
rgb = img[:, :, [3, 2, 1]]

# Normalización Min-Max para visualización
rgb_norm = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(rgb_norm)
plt.title("Imagen Sentinel-2 (RGB)")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap='viridis')
plt.title("Máscara (Ground Truth)")
plt.axis('off')

plt.tight_layout()
plt.show()